# COFY demo: a class-owned federation, live

This notebook runs a real **COFY** federation (Class-Owned Federated YOLO) inside Colab: one server process and
four client processes talking over HTTP, each client with its own small dataset and its own classes. You can
open the server dashboard and every client's page while it trains.

*Runtime → Change runtime type → T4 GPU* makes it faster; it also runs on the CPU runtime.
Run the cells in order.

In [ ]:
# 1. Code and dependencies (about 2 minutes)
BRANCH = "master"
!git clone -q --depth 1 -b {BRANCH} https://github.com/SalvatoreCapuozzo/DAFYOLO.git
%cd DAFYOLO/fedyolo_system
!pip install -q -r requirements.txt "ultralytics==8.4.84"

## 2. Choose a use case and a scheme

- `kfm12`: parasite eggs (12 classes) · `wbc`: white blood cells (7) · `dentex`: teeth on panoramic X-rays (32)
- `sequential`: the server schedules the turns, one node at a time · `concurrent`: every node trains at once and
  its upload is merged on arrival

In [ ]:
USE_CASE = "kfm12"     # kfm12 | wbc | dentex
SCHEME = "sequential"  # sequential | concurrent

import subprocess, sys, time, torch
extra = ["--scale", "l", "--imgsz", "640"] if torch.cuda.is_available() else []   # the study's model on a GPU
log = open("demo.log", "w")
demo = subprocess.Popen([sys.executable, "demo/run_demo.py", "--use-case", USE_CASE, "--scheme", SCHEME,
                         "--keep-open", *extra], stdout=log, stderr=subprocess.STDOUT)
time.sleep(25)
print(open("demo.log").read())

## 3. Open the dashboards

Each link opens in a new tab. On the server dashboard, open the room to see the turns, the uploads and the
calibration as they happen; each client page shows that node's classes, its training and its uploads.

In [ ]:
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8900, anchor_text="Server dashboard")
for i, node in enumerate("ABCD"):
    serve_kernel_port_as_window(8901 + i, anchor_text=f"Client page, node {node}")

## 4. Follow the federation

The cell prints the run as it goes: whose turn it is, each node's state, the calibration contributions and the
accuracy the server measures on a small evaluation set. With six images per node the accuracy only shows that
the pipeline works; the paper's results come from the full datasets.

In [ ]:
seen = 0
while True:
    text = open("demo.log").read()
    print(text[seen:], end="")
    seen = len(text)
    if "Done in" in text or demo.poll() is not None:
        break
    time.sleep(5)

## 5. Stop

The dashboards stay open until you run this cell. To try another use case or scheme, stop, then run step 2 again.

In [ ]:
demo.terminate(); demo.wait(timeout=30)
print("stopped")